# M5.4: Vector Index Management - Production Data Management

**Level 2 | Duration: ~32 minutes**

Learn to manage Pinecone vector indexes in production with backup, restoration, blue-green deployments, and migration strategies.

## 1. Introduction & Hook

**The Problem:**
Production vector indexes face real-world challenges:
- **Index corruption** from deployment errors or data issues
- **Model upgrades** requiring re-embedding millions of vectors
- **Zero-downtime deployments** for critical production systems
- **Disaster recovery** without proper backup strategies

**Current approaches lack:**
- Point-in-time recovery mechanisms
- Safe migration paths between embedding models
- Cost-aware deployment strategies

**What you'll build:**
Production-grade systems for backup/restore, blue-green deployments, and validated migrations—with realistic trade-offs and cost awareness.

In [ ]:
# Setup: Import core modules
import sys
import logging
from m5_4_vector_index.config import get_clients, validate_config
from m5_4_vector_index.core import (
    IndexBackupManager,
    BlueGreenDeploymentManager,
    IndexMigrationManager,
    CostOptimizationCalculator
)

logging.basicConfig(level=logging.INFO)

# Validate configuration
if not validate_config():
    print("⚠️ Configuration incomplete. Check .env file.")
else:
    print("✓ Configuration valid")

# Expected: Configuration validation message

## 2. Prerequisites & Setup

**Required Background:**
- **Level 1 Modules:** M1.1, M1.2 (vector database basics)
- **M5.1-M5.3:** Incremental updates, data validation, monitoring

**New Dependencies:**
- `boto3` - AWS S3 operations for backups
- `redis` - Blue-green traffic coordination
- `tqdm` - Progress tracking for large operations

**What you need:**
- Pinecone API key (required)
- AWS credentials (optional, for backup functionality)
- Redis instance (optional, for blue-green deployments)

In [ ]:
# Initialize all clients
clients = get_clients()

print("=== Client Status ===")
print(f"Pinecone:  {'✓ Available' if clients['pinecone'] else '✗ Unavailable'}")
print(f"S3:        {'✓ Available' if clients['s3'] else '✗ Unavailable (backup disabled)'}")
print(f"Redis:     {'✓ Available' if clients['redis'] else '✗ Unavailable (blue-green disabled)'}")

# Expected: Status report showing which clients are configured

## 3. Theory Foundation

### Three Core Patterns

**1. Backup & Restore**
> "Think of this like Git for your vector index. You export all vectors to durable storage (S3), verify data integrity, and can restore to any point in time."

**Key concepts:**
- Batch fetching (1000 vectors at a time)
- gzip compression (level 6)
- MD5 checksum validation
- S3 metadata storage

**2. Blue-Green Deployment**
> "Named after the two environments (blue = current, green = new), this pattern lets you build a completely new index version while the old one serves traffic."

**Key concepts:**
- Parallel index creation
- Redis-based traffic coordination
- Atomic switching
- Instant rollback capability

**3. Migration with Verification**
> "Moving between indexes requires careful copying, integrity validation, and handling transformation logic."

**Key concepts:**
- Optional vector transformation (re-embedding)
- Dimension validation
- Sampling-based verification
- Progress tracking

In [ ]:
# Visualize the three patterns
print("=== Pattern Comparison ===\n")

patterns = {
    "Backup & Restore": {
        "Use case": "Disaster recovery, rollback",
        "Storage": "S3 durable storage",
        "Downtime": "Minutes to hours",
        "Cost": "Low"
    },
    "Blue-Green Deployment": {
        "Use case": "Zero-downtime updates",
        "Storage": "Double index temporarily",
        "Downtime": "Zero",
        "Cost": "High (2x storage)"
    },
    "Migration": {
        "Use case": "Model upgrades, transformations",
        "Storage": "Source + target",
        "Downtime": "Depends on verification",
        "Cost": "Medium to high"
    }
}

for name, props in patterns.items():
    print(f"[{name}]")
    for key, value in props.items():
        print(f"  {key}: {value}")
    print()

# Expected: Table comparing the three patterns

## 4. Hands-On Implementation

### Step 1: Automated Backup System

The `IndexBackupManager` handles:
- Fetching vectors from Pinecone in batches (1000 at a time)
- Compression using gzip (level 6)
- MD5 checksum calculation for integrity verification
- S3 upload with metadata storage
- Namespaced backups

**Key features:**
- Progress tracking with tqdm
- Batch processing to avoid memory overload
- Checksum validation for corruption detection

In [ ]:
# Step 1: Backup Manager Demo
from m5_4_vector_index.config import Config

if clients['s3']:
    backup_mgr = IndexBackupManager(
        clients['pinecone'],
        clients['s3'],
        Config.S3_BACKUP_BUCKET,
        batch_size=1000
    )
    
    print("✓ Backup manager initialized")
    print(f"  Bucket: {Config.S3_BACKUP_BUCKET}")
    print(f"  Batch size: 1000 vectors")
    
    # List existing backups (safe to call without live index)
    print("\n[List Backups]")
    backups = backup_mgr.list_backups(index_name="example-index")
    print(f"Found {len(backups)} existing backups")
    
    # Backup example (commented - requires live index)
    # metadata = backup_mgr.backup_index("my-index", namespace="production")
    # print(f"Backup created: {metadata.backup_id}")
else:
    print("⚠️ Skipping backup demo (no S3 credentials)")

# Expected: Backup manager status or skip message

### Step 2: Blue-Green Deployment Manager

Creates parallel index versions for zero-downtime updates:

**Workflow:**
1. Create new "green" index with updated configuration
2. Populate green while blue serves production traffic
3. Coordinate traffic switch via Redis flag
4. Monitor green for issues with instant rollback capability

**Benefits:**
- Zero downtime during deployments
- Instant rollback if issues detected
- A/B testing between index versions

**Trade-offs:**
- Requires double storage temporarily
- Redis dependency for coordination

In [ ]:
# Step 2: Blue-Green Deployment Demo

if clients['redis']:
    bg_mgr = BlueGreenDeploymentManager(clients['pinecone'], clients['redis'])
    
    print("✓ Blue-green manager initialized")
    print("  Traffic coordination: Redis")
    
    # Simulate deployment workflow
    blue_index = "my-index-blue"
    green_index = "my-index-green"
    
    print(f"\n[Deployment Workflow]")
    print(f"1. Blue (current): {blue_index}")
    print(f"2. Green (new):    {green_index}")
    print(f"3. Switch:         bg_mgr.switch_traffic(blue, green)")
    print(f"4. Rollback:       bg_mgr.rollback(blue)")
    
    # Get active index (safe to call)
    active = bg_mgr.get_active_index(blue_index)
    print(f"\nCurrent active: {active}")
    
    # Actual operations (commented - require live indexes)
    # bg_mgr.create_green_index(blue_index, green_index, dimension=1536)
    # bg_mgr.switch_traffic(blue_index, green_index)
    # bg_mgr.rollback(blue_index)
else:
    print("⚠️ Skipping blue-green demo (no Redis connection)")

# Expected: Blue-green workflow description or skip message

### Step 3: Migration Manager with Verification

Handles transformations between indexes:

**Features:**
- Export from source with optional re-embedding
- Validate transformed vectors (dimension checks, quality metrics)
- Import with progress tracking
- Verify via sampling queries against both indexes

**Use cases:**
- Upgrading to new embedding model (e.g., OpenAI ada-002 → ada-003)
- Changing vector dimensions
- Reprocessing with improved metadata

**Verification strategy:**
- Sample-based (default 100 vectors)
- Checks existence in both indexes
- Not 100% coverage (trade-off for speed)

In [ ]:
# Step 3: Migration Manager Demo

if clients['pinecone']:
    migration_mgr = IndexMigrationManager(clients['pinecone'], batch_size=1000)
    
    print("✓ Migration manager initialized")
    print("  Batch size: 1000 vectors")
    print("  Default verification: 100 samples")
    
    # Example transformation function
    def scale_vectors(vector_dict):
        \"\"\"Example: Scale all vector values by 0.9\"\"\"
        vector_dict['values'] = [v * 0.9 for v in vector_dict['values']]
        return vector_dict
    
    print("\n[Migration Workflow]")
    print("1. Define transformation (optional)")
    print("2. Migrate: migration_mgr.migrate_index(source, target, transform_fn)")
    print("3. Verification runs automatically on sample")
    
    # Migration example (commented - requires live indexes)
    # result = migration_mgr.migrate_index(
    #     "source-index",
    #     "target-index",
    #     transform_fn=scale_vectors,
    #     verify_sample_size=100
    # )
    # print(f"Migrated: {result.vectors_migrated}, Verified: {result.vectors_verified}")
else:
    print("⚠️ Skipping migration demo (no Pinecone client)")

# Expected: Migration workflow description

### Step 4: Cost Optimization Calculator

Tracks costs across:
- **Storage:** Per vector count and dimension
- **API calls:** Queries and upserts
- **Data transfer:** Network egress fees
- **Backup storage:** S3 fees

**Pricing assumptions (as of 2024):**
- $0.10 per 1M queries
- $0.20 per 1M upserts
- $0.30 per GB/month storage
- $0.09 per GB data transfer

**Example:** For 500K vectors at 1536 dimensions running a 7-day blue-green deployment.

In [ ]:
# Step 4: Cost Optimization Demo
from m5_4_vector_index.config import Config

cost_calc = CostOptimizationCalculator(
    cost_per_million_queries=Config.COST_PER_MILLION_QUERIES,
    cost_per_million_upserts=Config.COST_PER_MILLION_UPSERTS,
    cost_per_gb_storage_monthly=Config.COST_PER_GB_STORAGE_MONTHLY,
    cost_per_gb_transfer=Config.COST_PER_GB_TRANSFER
)

print("✓ Cost calculator initialized\n")

# Example: Estimate migration cost for 500K vectors
breakdown = cost_calc.estimate_migration_cost(
    vector_count=500_000,
    dimension=1536,
    days_dual_index=7  # 7 days running both blue and green
)

cost_calc.print_cost_breakdown(breakdown)

print(f"\n[Key Insight]")
print(f"Blue-green deployment for 500K vectors (7 days): ${breakdown.total_cost:.2f}")
print(f"Largest cost component: Storage (double indexing)")

# Expected: Detailed cost breakdown table

## 5. Reality Check

### What This DOESN'T Do

**Limitations:**
- ❌ Real-time synchronization between blue and green
- ❌ Automatic rollback detection
- ❌ Schema evolution beyond vector dimension changes
- ❌ Sparse vector transformations automatically

### Trade-offs Accepted

**Cost vs. Availability:**
- Blue-green requires double storage temporarily
- Backup/restore adds latency to disaster recovery workflows
- Migration verification requires sampling (not 100% coverage)

### When This Approach Breaks

**Breaking points:**
1. **Extremely large indexes (>10M vectors)** with tight SLAs may timeout
2. **Frequent schema changes** (daily/weekly model updates) make blue-green costly
3. **Cost-constrained environments** need simpler alternatives
4. **Network interruptions** during large transfers (>10GB) require resumable uploads

In [ ]:
# Reality Check: Calculate breaking points

print("=== Breaking Point Analysis ===\n")

# Small index: overhead exceeds benefits
small_breakdown = cost_calc.estimate_migration_cost(10_000, 1536, 7)
print(f"Small index (10K vectors, 7 days):")
print(f"  Cost: ${small_breakdown.total_cost:.2f}")
print(f"  Verdict: Overhead likely exceeds benefits\n")

# Medium index: sweet spot
medium_breakdown = cost_calc.estimate_migration_cost(500_000, 1536, 7)
print(f"Medium index (500K vectors, 7 days):")
print(f"  Cost: ${medium_breakdown.total_cost:.2f}")
print(f"  Verdict: Good candidate for blue-green\n")

# Large index: potential timeout risk
large_breakdown = cost_calc.estimate_migration_cost(10_000_000, 1536, 7)
print(f"Large index (10M vectors, 7 days):")
print(f"  Cost: ${large_breakdown.total_cost:.2f}")
print(f"  Verdict: Risk of timeout during verification\n")

# Expected: Cost analysis showing when blue-green makes sense

## 6. Alternative Solutions

### Alternative 1: Single Index with Versioned Metadata

**Approach:** Use metadata tagging instead of separate indexes.

**Pros:**
- Simpler implementation
- Lower storage costs
- No traffic switching logic

**Cons:**
- No true zero-downtime capability
- Harder to rollback
- Query performance impact from metadata filtering

**Best for:** Small to medium indexes where brief downtime is acceptable

---

### Alternative 2: Managed Services with Built-in Backup

**Approach:** Some vector databases offer native replication and backup.

**Pros:**
- Operational simplicity
- Vendor-supported
- Built-in monitoring

**Cons:**
- Vendor lock-in
- Less control over backup format
- Potentially higher costs

**Best for:** Teams prioritizing operational simplicity over control

---

### Alternative 3: Vector Database with Native Replication

**Approach:** Technologies like Weaviate offer built-in high-availability.

**Pros:**
- Reduces custom code
- Battle-tested replication
- Automatic failover

**Cons:**
- Platform-specific
- Migration between platforms harder
- Learning curve for new technology

**Best for:** New projects not locked into Pinecone

---

### Alternative 4: Just Reindex from Source (No Backups)

**Approach:** \"For systems where source documents are always available and index corruption is acceptable, skip backups entirely.\"

**Pros:**
- Simplest approach
- No backup storage costs
- Fast iteration

**Cons:**
- Risky in production
- Reindex time = recovery time
- Loss of historical point-in-time recovery

**Best for:** Development/staging environments, or systems with fast reindexing (<1 hour)

In [ ]:
# Alternative Solutions Comparison

print("=== Solution Comparison ===\n")

alternatives = [
    {
        "name": "Blue-Green (This Module)",
        "complexity": "High",
        "cost": "High",
        "downtime": "Zero",
        "best_for": "Critical production systems"
    },
    {
        "name": "Versioned Metadata",
        "complexity": "Low",
        "cost": "Low",
        "downtime": "Minutes",
        "best_for": "Small-medium indexes"
    },
    {
        "name": "Managed Services",
        "complexity": "Low",
        "cost": "Medium-High",
        "downtime": "Zero",
        "best_for": "Simplicity-first teams"
    },
    {
        "name": "Just Reindex",
        "complexity": "Very Low",
        "cost": "Very Low",
        "downtime": "Hours",
        "best_for": "Dev/staging only"
    }
]

for alt in alternatives:
    print(f"[{alt['name']}]")
    print(f"  Complexity: {alt['complexity']}")
    print(f"  Cost:       {alt['cost']}")
    print(f"  Downtime:   {alt['downtime']}")
    print(f"  Best for:   {alt['best_for']}")
    print()

# Expected: Comparison table of alternatives

## 7. When NOT to Use

### Avoid Blue-Green Deployments For:

**1. Small Indexes (<10,000 vectors)**
- Overhead exceeds benefits
- Simple reindex is faster
- Cost savings minimal

**2. Cost-Constrained Budgets (<$200/month)**
- Double indexing is uneconomical
- Simpler alternatives sufficient
- Better to invest in other areas

**3. Frequently Changing Schemas**
- Daily/weekly model updates make migration constant
- Better to use versioned metadata
- Or accept brief downtime for updates

**4. Massive Indexes (>10M vectors) with Tight SLAs**
- Risk of timeout during verification
- Consider incremental migration strategies
- Or use managed services with built-in replication

### Better Alternatives When:

- **Development/Staging:** Just reindex from source
- **Small teams:** Metadata versioning or managed services
- **Infrequent updates:** Scheduled maintenance windows
- **Budget-first:** Backup/restore without blue-green

In [ ]:
# Decision Helper: Should you use blue-green?

def should_use_blue_green(vector_count, budget_monthly, update_frequency_days, sla_requirement):
    \"\"\"
    Decision helper for blue-green deployment.
    
    Args:
        vector_count: Number of vectors in index
        budget_monthly: Monthly budget in USD
        update_frequency_days: How often updates occur (in days)
        sla_requirement: Required uptime (e.g., "99.9%")
    
    Returns:
        (bool, str): (recommendation, reason)
    \"\"\"
    if vector_count < 10_000:
        return False, "Index too small - overhead exceeds benefits"
    
    if vector_count > 10_000_000 and sla_requirement == "99.99%":
        return False, "Index too large with tight SLA - risk of timeout"
    
    if budget_monthly < 200:
        return False, "Budget too constrained for double indexing"
    
    if update_frequency_days < 7:
        return False, "Too frequent updates - consider metadata versioning"
    
    return True, "Good candidate for blue-green deployment"

# Test scenarios
scenarios = [
    (50_000, 500, 30, "99.9%"),
    (5_000, 100, 30, "99%"),
    (15_000_000, 2000, 90, "99.99%"),
    (500_000, 1000, 14, "99.9%")
]

print("=== Blue-Green Decision Helper ===\n")
for vectors, budget, freq, sla in scenarios:
    should_use, reason = should_use_blue_green(vectors, budget, freq, sla)
    print(f"{vectors:,} vectors, ${budget}/mo, updates every {freq}d")
    print(f"  Recommendation: {'✓ USE' if should_use else '✗ SKIP'}")
    print(f"  Reason: {reason}\n")

# Expected: Decision recommendations for various scenarios

## 8. Common Failures

### Failure 1: Backup Corruption During Large Transfers (>10GB)

**Symptom:** MD5 checksum mismatch on restore

**Cause:** Network interruptions during S3 upload

**Solution:**
- Implement resumable uploads (S3 multipart)
- Multi-part transfer verification
- Retry logic with exponential backoff

---

### Failure 2: Index Switch Timing Issues Causing Downtime

**Symptom:** Requests hitting intermediate state during traffic switch

**Cause:** Race condition between Redis flag update and application reads

**Solution:**
- Use connection pooling with atomic flag updates
- Implement health checks before switching
- Add graceful degradation for transition period

---

### Failure 3: Migration Data Loss from Partial Failures

**Symptom:** Process dies mid-upsert batch, leaving incomplete migration

**Cause:** Crash, network timeout, or API rate limit

**Solution:**
- Track batch offsets in persistent storage
- Resume from last successful checkpoint
- Implement idempotent upserts (safe to retry)

---

### Failure 4: Cost Spike During Migration (Double Indexing)

**Symptom:** Unexpected cloud bill for running both indexes

**Cause:** Forgot to clean up old index after migration

**Solution:**
- Schedule migrations during off-peak hours
- Use cheaper index tiers temporarily (if available)
- Automated cleanup with retention policies
- Cost alerts in monitoring

---

### Failure 5: Rollback Failures (No Working Previous Version)

**Symptom:** Can't rollback because only most recent backup exists

**Cause:** Only keeping one backup, which might be corrupted

**Solution:**
- Maintain 3+ versioned backups with rotation policy
- Test restores in staging monthly
- Document rollback procedures
- Automated backup validation

In [ ]:
# Common Failures: Demonstration of fixes

print("=== Failure Scenarios & Fixes ===\n")

# Scenario 1: Checksum verification
print("[Scenario 1: Backup Corruption]")
print("Problem: Network interruption during 15GB backup upload")
print("Fix: Use S3 multipart upload with part-level checksums")
print("Code: s3.upload_fileobj() with TransferConfig(multipart_threshold=100MB)")
print()

# Scenario 2: Graceful traffic switching
print("[Scenario 2: Traffic Switch Race Condition]")
print("Problem: Some requests hit old index during switch")
print("Fix: Connection pooling + health check before switch")
print("Code:")
print("  1. redis.set('active_index', green_index)  # Atomic")
print("  2. Health check green index")
print("  3. Wait for connection pool refresh (5-10s)")
print()

# Scenario 3: Resume from checkpoint
print("[Scenario 3: Migration Partial Failure]")
print("Problem: Process crashes at batch 450/1000")
print("Fix: Track progress in persistent storage")
print("Code:")
print("  redis.set(f'migration:{id}:last_batch', batch_num)")
print("  # On restart: resume from last_batch + 1")
print()

# Scenario 4: Cost monitoring
print("[Scenario 4: Cost Spike from Double Indexing]")
print("Problem: Blue + green running for 30 days (forgot cleanup)")
expected_cost_7d = cost_calc.estimate_migration_cost(500_000, 1536, 7).total_cost
expected_cost_30d = cost_calc.estimate_migration_cost(500_000, 1536, 30).total_cost
print(f"  7-day migration:  ${expected_cost_7d:.2f}")
print(f"  30-day (forgot):  ${expected_cost_30d:.2f}")
print(f"  Waste:            ${expected_cost_30d - expected_cost_7d:.2f}")
print("Fix: Automated cleanup after successful verification")
print()

# Scenario 5: Backup rotation
print("[Scenario 5: Rollback with Corrupted Backup]")
print("Problem: Only 1 backup exists, and it's corrupted")
print("Fix: Keep 3+ backups with rotation")
print("Policy: Daily backups, keep latest 7, weekly for 4 weeks, monthly for 12 months")

# Expected: Failure scenarios with concrete fixes

## 9. Production Considerations

### Scaling Concerns

**Batch Size Optimization:**
- Too small: Slow migration, high API call count
- Too large: Memory pressure, timeout risk
- Sweet spot: 500-1000 vectors per batch

**Network Bandwidth:**
- 1M vectors @ 1536 dims ≈ 7GB compressed
- Plan for multi-hour transfers
- Consider region proximity (S3 → Pinecone)

**Pinecone API Rate Limits:**
- Queries: Varies by plan (typically 200-1000 QPS)
- Upserts: Similar limits apply
- Implement exponential backoff on 429 errors

**Backup Storage Growth:**
- Daily backups accumulate quickly
- Example: 500K vectors @ 1536 dims = ~900MB/backup
- 30 days = ~27GB storage costs

---

### Monitoring Requirements

**Backup Metrics:**
- ✓ Backup completion times
- ✓ Backup file sizes
- ✓ Checksum validation success rate
- ✓ S3 upload errors

**Migration Metrics:**
- ✓ Vectors migrated per second
- ✓ Error rate per batch
- ✓ Verification sample pass rate
- ✓ End-to-end duration

**Blue-Green Metrics:**
- ✓ Traffic distribution (% blue vs. green)
- ✓ Query latency on both indexes
- ✓ Error rates pre/post switch
- ✓ Rollback frequency

**Cost Tracking:**
- ✓ Daily storage costs
- ✓ API call costs (queries + upserts)
- ✓ Data transfer fees
- ✓ Budget alerts

---

### Production Deployment Checklist

- [ ] Automated backup scheduling (daily minimum)
- [ ] Restore testing in staging environment (monthly)
- [ ] Blue-green rollback procedures documented
- [ ] Cost tracking dashboard active
- [ ] Alerting on backup failures (<5 min notification)
- [ ] Rate limit handling with exponential backoff
- [ ] Runbook for common failure scenarios
- [ ] Access control for production indexes
- [ ] Audit logging for all index operations
- [ ] Disaster recovery plan tested quarterly

In [ ]:
# Production Scaling Calculations

print("=== Production Scaling Analysis ===\n")

# Batch size impact
def estimate_migration_time(vector_count, batch_size, api_latency_ms=100):
    \"\"\"Estimate migration duration based on batch size.\"\"\"
    total_batches = vector_count / batch_size
    total_seconds = (total_batches * api_latency_ms) / 1000
    return total_seconds / 60  # minutes

batch_sizes = [100, 500, 1000, 2000]
print("[Batch Size Impact - 500K vectors]")
for batch in batch_sizes:
    time_min = estimate_migration_time(500_000, batch)
    print(f"  {batch:4d} vectors/batch: ~{time_min:.1f} min")

print()

# Storage growth
print("[Backup Storage Growth - 500K vectors @ 1536 dims]")
backup_size_mb = (500_000 * 1536 * 4 * 1.2) / (1024**2) * 0.3  # 30% compression
print(f"  Per backup:   ~{backup_size_mb:.0f} MB")
print(f"  Daily (30d):  ~{backup_size_mb * 30 / 1024:.1f} GB")
print(f"  Monthly (12): ~{backup_size_mb * 12 / 1024:.1f} GB")

print()

# Rate limit impact
print("[API Rate Limit Impact]")
qps_limits = [100, 500, 1000]
for qps in qps_limits:
    vectors_per_hour = qps * 3600  # 1 query per batch
    time_hours = 500_000 / vectors_per_hour
    print(f"  {qps:4d} QPS: {time_hours:.1f} hours for full migration")

print()

# Cost monitoring thresholds
print("[Cost Alert Thresholds]")
thresholds = {
    "Warning": 100,
    "Critical": 500,
    "Emergency": 1000
}
for level, cost in thresholds.items():
    print(f"  {level:10s}: Alert if daily cost > ${cost}")

# Expected: Production scaling metrics and calculations

## 10. Decision Card

Quick reference for choosing vector index management strategies:

| Scenario | Best Approach | Cost | Complexity | Downtime |
|----------|---------------|------|-----------|----------|
| **Simple backup only** | S3 backup/restore | Low | Low | Minutes-Hours |
| **Zero-downtime deployment** | Blue-green | High | High | Zero |
| **Model upgrade** | Blue-green + migration | High | Very High | Zero |
| **Disaster recovery** | Backup rotation | Medium | Medium | Minutes-Hours |
| **Cost-constrained** | Metadata versioning | Low | Low | Minutes |
| **Dev/Staging** | Just reindex | Very Low | Very Low | Hours (acceptable) |

---

### Decision Tree

```
START
  ├─ Need zero downtime?
  │   ├─ YES → Check budget
  │   │   ├─ >$500/month → Blue-Green
  │   │   └─ <$500/month → Metadata versioning
  │   └─ NO → Check recovery time
  │       ├─ <1 hour needed → Backup/restore
  │       └─ >1 hour OK → Just reindex
  │
  ├─ Model transformation needed?
  │   ├─ YES → Migration with verification
  │   └─ NO → Simple backup/restore
  │
  └─ Index size
      ├─ <10K vectors → Skip blue-green
      ├─ 10K-10M vectors → Blue-green OK
      └─ >10M vectors → Incremental migration
```

---

### Key Takeaways

1. **Blue-green is expensive** - Only use when zero downtime is critical
2. **Backup everything** - Even if you have blue-green, backups are insurance
3. **Test restores regularly** - Untested backups are worthless
4. **Monitor costs** - Double indexing can surprise you
5. **Document rollback** - Pressure situations need clear procedures

In [ ]:
# Interactive Decision Card

def recommend_strategy(
    vector_count: int,
    zero_downtime_required: bool,
    budget_monthly: int,
    recovery_time_max_hours: float,
    transformation_needed: bool
):
    \"\"\"
    Recommend index management strategy based on requirements.
    
    Returns:
        (str, str): (strategy, reasoning)
    \"\"\"
    # Size check first
    if vector_count < 10_000:
        return "Just Reindex or Backup/Restore", "Index too small for complex strategies"
    
    # Zero downtime path
    if zero_downtime_required:
        if budget_monthly < 500:
            return "Metadata Versioning", "Zero downtime needed but budget constrained"
        else:
            if transformation_needed:
                return "Blue-Green + Migration", "Zero downtime with model transformation"
            else:
                return "Blue-Green Deployment", "Zero downtime without transformation"
    
    # No zero downtime required
    if transformation_needed:
        return "Migration with Verification", "Transformation needed, downtime acceptable"
    
    if recovery_time_max_hours < 1:
        return "Backup/Restore with frequent backups", "Fast recovery needed"
    else:
        return "Simple Backup/Restore or Just Reindex", "Flexible recovery time"

# Test interactive decision
print("=== Strategy Recommender ===\n")

test_cases = [
    {
        "desc": "Startup (50K vectors, tight budget)",
        "vector_count": 50_000,
        "zero_downtime": False,
        "budget": 200,
        "recovery_hours": 4,
        "transform": False
    },
    {
        "desc": "Production SaaS (1M vectors, critical uptime)",
        "vector_count": 1_000_000,
        "zero_downtime": True,
        "budget": 2000,
        "recovery_hours": 0,
        "transform": False
    },
    {
        "desc": "Model upgrade (500K vectors, new embeddings)",
        "vector_count": 500_000,
        "zero_downtime": True,
        "budget": 1000,
        "recovery_hours": 0,
        "transform": True
    }
]

for case in test_cases:
    strategy, reason = recommend_strategy(
        case["vector_count"],
        case["zero_downtime"],
        case["budget"],
        case["recovery_hours"],
        case["transform"]
    )
    print(f"[{case['desc']}]")
    print(f"  Recommendation: {strategy}")
    print(f"  Reasoning: {reason}\n")

# Expected: Personalized strategy recommendations

## 11. Wrap-Up & Next Steps

### What You've Learned

✓ **Backup & Restore** - S3-backed disaster recovery with integrity verification  \n✓ **Blue-Green Deployment** - Zero-downtime index updates with instant rollback  \n✓ **Migration Strategies** - Safe vector transformations with sampling verification  \n✓ **Cost Optimization** - Financial tracking and decision frameworks  \n✓ **Failure Handling** - Common production issues and concrete fixes\n\n---\n\n### Integration with M5.1-M5.3\n\n**M5.1 (Incremental Updates)** → Use backup before bulk updates  \n**M5.2 (Data Validation)** → Validate vectors pre-migration  \n**M5.3 (Monitoring)** → Track backup/migration metrics  \n**M5.4 (This Module)** → Complete production data management\n\n---\n\n### Deployment to Railway\n\n**Steps:**\n1. Push code to GitHub repository\n2. Connect Railway to your repo\n3. Set environment variables (PINECONE_API_KEY, AWS credentials, Redis URL)\n4. Deploy `app.py` as web service\n5. Schedule backup jobs using Railway Cron or external scheduler\n\n**Environment Variables Required:**\n```\nPINECONE_API_KEY=...\nAWS_ACCESS_KEY_ID=...\nAWS_SECRET_ACCESS_KEY=...\nREDIS_HOST=...\nS3_BACKUP_BUCKET=...\n```\n\n---\n\n### Monitoring Dashboard Integration\n\n**Key Metrics to Track:**\n- Backup success rate (target: >99%)\n- Restore test frequency (monthly minimum)\n- Blue-green deployment duration\n- Migration error rates\n- Cost per day (with budget alerts)\n\n**Tools:**\n- Prometheus + Grafana for metrics\n- Sentry for error tracking\n- CloudWatch for AWS S3 monitoring\n- Custom dashboards for cost tracking\n\n---\n\n### Next Module: M6 (Security & Compliance)\n\n**Topics:**\n- Access control for vector indexes\n- Encryption at rest and in transit\n- Audit logging for compliance (GDPR, SOC2)\n- PII handling in vector metadata\n- Secrets management for API keys\n\n---\n\n### Practathon Challenges\n\n**🟢 EASY (90 minutes):**  \nImplement basic backup and restore for a single namespace with checksum verification.\n\n**🟡 MEDIUM (2-3 hours):**  \nBuild blue-green deployment with Redis coordination and automated traffic switching.\n\n**🔴 HARD (6-8 hours):**  \nFull migration system with re-embedding transformation, verification sampling, and cost tracking.\n\n---\n\n### Resources\n\n- [Pinecone Docs](https://docs.pinecone.io/)\n- [AWS S3 Best Practices](https://docs.aws.amazon.com/AmazonS3/latest/userguide/best-practices.html)\n- [Blue-Green Deployment Pattern](https://martinfowler.com/bliki/BlueGreenDeployment.html)\n- [This Module's Code Repository](https://github.com/yesvisare/ccc_l2_aug_practical)\n\n---\n\n**🎉 Congratulations!** You now have production-grade vector index management skills."

In [ ]:
# Module Summary

print("=== M5.4: Vector Index Management - Complete ===\\n")

print("[What You Built]")
print("✓ IndexBackupManager - S3-backed backups with checksums")
print("✓ BlueGreenDeploymentManager - Zero-downtime deployments")
print("✓ IndexMigrationManager - Vector transformations")
print("✓ CostOptimizationCalculator - Financial tracking")

print("\\n[Key Metrics from This Session]")
if 'breakdown' in dir():
    print(f"Example cost (500K vectors, 7d): ${breakdown.total_cost:.2f}")
if 'clients' in dir():
    active_clients = sum(1 for c in clients.values() if c is not None)
    print(f"Active clients: {active_clients}/3 (Pinecone, S3, Redis)")

print("\\n[Next Steps]")
print("1. Set up .env file with your credentials")
print("2. Test backup/restore with a small index")
print("3. Deploy to Railway with scheduled backups")
print("4. Integrate monitoring dashboard")
print("5. Proceed to M6 (Security & Compliance)")

print("\\n[Quick Start Commands]")
print("  python config.py              # Validate configuration")
print("  python l2_m4_vector_index_management.py  # Run examples")
print("  uvicorn app:app --reload      # Start API server")

print("\\n✓ Module M5.4 completed successfully!")

# Expected: Summary of module completion and next steps